# Real-ESRGAN Video Upscaler for Kaggle GPU

A simplified **single-GPU** version of the original notebook you uploaded.

### What this notebook does
1. You upload your video to the Kaggle notebook with **Add Data → Upload**.
2. You set the video path in one cell.
3. Real-ESRGAN runs on Kaggle's NVIDIA GPU.
4. The original FPS is detected automatically.
5. The original audio is copied into the final MP4 when possible.
6. A download link is shown at the end.

**Recommended:** `SCALE = 2` for 1080p → 4K.  
Use `SCALE = 4` for lower-resolution sources such as 720p → 4K.

Your local GT 710 is not used for the AI processing.

## 1. Settings

First use Kaggle's **Add Data → Upload** to upload your video. Then change only `INPUT_VIDEO` below to the path of the uploaded file.

In [ ]:
# ===== USER SETTINGS =====

# Change this to your uploaded video's Kaggle path.
# Example: /kaggle/input/my-video/video.mp4
INPUT_VIDEO = "/kaggle/input/your-video/video.mp4"

# 2 = recommended for 1080p -> 4K
# 4 = useful for lower-resolution sources such as 720p
SCALE = 2

# Normal general-purpose Real-ESRGAN model
MODEL = "RealESRGAN_x4plus"

# Final H.264 quality. Lower CRF = larger/higher-quality file.
CRF = 18
PRESET = "medium"

OUTPUT_VIDEO = "/kaggle/working/upscaled_video.mp4"
WORK_DIR = "/kaggle/working/realesrgan_video"

print("Input:", INPUT_VIDEO)
print("Scale:", SCALE)
print("Output:", OUTPUT_VIDEO)

## 2. Check Kaggle GPU

This cell must show an NVIDIA GPU. If it says CUDA is unavailable, enable **Settings → Accelerator → GPU** in Kaggle.

In [ ]:
import os
import subprocess
import torch

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if not torch.cuda.is_available():
    raise RuntimeError(
        "No NVIDIA GPU detected. In Kaggle open Settings -> Accelerator and select GPU."
    )

print("GPU:", torch.cuda.get_device_name(0))
print("VRAM (GB):", round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2))
subprocess.run(["nvidia-smi"], check=False)

## 3. Install Real-ESRGAN

This uses the official Real-ESRGAN repository. The old notebook had several version pins and two-GPU logic that are unnecessary here.

In [ ]:
%pip install -q --upgrade pip
%pip install -q basicsr facexlib gfpgan opencv-python-headless
!rm -rf /kaggle/working/Real-ESRGAN
!git clone --depth 1 https://github.com/xinntao/Real-ESRGAN.git /kaggle/working/Real-ESRGAN
%pip install -q -r /kaggle/working/Real-ESRGAN/requirements.txt
%pip install -q -e /kaggle/working/Real-ESRGAN

## 4. Download the Real-ESRGAN model weights

In [ ]:
import os
import urllib.request

REPO = "/kaggle/working/Real-ESRGAN"
WEIGHTS_DIR = os.path.join(REPO, "weights")
os.makedirs(WEIGHTS_DIR, exist_ok=True)

weights_path = os.path.join(WEIGHTS_DIR, "RealESRGAN_x4plus.pth")
weights_url = "https://github.com/xinntao/Real-ESRGAN/releases/download/v0.2.5.0/RealESRGAN_x4plus.pth"

if not os.path.exists(weights_path):
    print("Downloading RealESRGAN_x4plus.pth...")
    urllib.request.urlretrieve(weights_url, weights_path)
else:
    print("Model already exists.")

print("Model size (MB):", round(os.path.getsize(weights_path) / 1024**2, 1))

## 5. Inspect the input video

In [ ]:
import json
import subprocess
import os

if not os.path.isfile(INPUT_VIDEO):
    raise FileNotFoundError(
        f"Video not found: {INPUT_VIDEO}\n"
        "Use Kaggle's Add Data -> Upload and update INPUT_VIDEO in the Settings cell."
    )

probe = subprocess.run(
    [
        "ffprobe", "-v", "error", "-select_streams", "v:0",
        "-show_entries",
        "stream=width,height,r_frame_rate,avg_frame_rate,nb_frames,duration",
        "-of", "json", INPUT_VIDEO
    ],
    capture_output=True, text=True, check=True
)

info = json.loads(probe.stdout)["streams"][0]

print("Resolution:", info.get("width"), "x", info.get("height"))
print("FPS:", info.get("avg_frame_rate") or info.get("r_frame_rate"))
print("Duration:", info.get("duration"), "seconds")
print("Frames:", info.get("nb_frames", "unknown"))

## 6. Extract frames

Frames are kept lossless as PNG during the AI stage. This is temporary Kaggle working storage.

In [ ]:
import os
import subprocess

FRAMES_DIR = os.path.join(WORK_DIR, "frames")
UPSCALED_DIR = os.path.join(WORK_DIR, "upscaled")

os.makedirs(FRAMES_DIR, exist_ok=True)
os.makedirs(UPSCALED_DIR, exist_ok=True)

subprocess.run(
    [
        "ffmpeg", "-y",
        "-i", INPUT_VIDEO,
        "-vsync", "0",
        os.path.join(FRAMES_DIR, "frame%08d.png")
    ],
    check=True
)

count = len([f for f in os.listdir(FRAMES_DIR) if f.endswith(".png")])
print(f"Extracted {count} frames")

## 7. AI upscale on the Kaggle GPU

This is the main processing step. It uses **one CUDA GPU** and automatically processes the whole frame directory.

`tile=256` is intentionally conservative so the notebook is less likely to run out of VRAM. If you have a large GPU and want more speed, you can later try `--tile 0`.

In [ ]:
import subprocess
import torch

if not torch.cuda.is_available():
    raise RuntimeError("CUDA is unavailable. Enable a Kaggle GPU first.")

cmd = [
    "python",
    f"{REPO}/inference_realesrgan.py",
    "-n", MODEL,
    "-i", FRAMES_DIR,
    "-o", UPSCALED_DIR,
    "-s", str(SCALE),
    "--suffix", "out",
    "--tile", "256",
    "--tile_pad", "10",
    "--fp32"
]

print("Starting Real-ESRGAN on:", torch.cuda.get_device_name(0))
subprocess.run(cmd, check=True)

count = len([f for f in os.listdir(UPSCALED_DIR) if f.endswith(".png")])
print(f"Upscaled {count} frames")

## 8. Rebuild the MP4 and preserve audio

The original notebook hardcoded 29.97 FPS. This version reads the actual source FPS, so 24/25/30/60 FPS videos keep their timing.

In [ ]:
import subprocess
import os

fps_result = subprocess.run(
    [
        "ffprobe", "-v", "error", "-select_streams", "v:0",
        "-show_entries", "stream=avg_frame_rate",
        "-of", "default=noprint_wrappers=1:nokey=1",
        INPUT_VIDEO
    ],
    capture_output=True, text=True, check=True
)

fps = fps_result.stdout.strip()
frame_pattern = os.path.join(UPSCALED_DIR, "frame%08d_out.png")

cmd = [
    "ffmpeg", "-y",
    "-framerate", fps,
    "-i", frame_pattern,
    "-i", INPUT_VIDEO,
    "-map", "0:v:0",
    "-map", "1:a?",
    "-c:v", "libx264",
    "-crf", str(CRF),
    "-preset", PRESET,
    "-pix_fmt", "yuv420p",
    "-c:a", "copy",
    "-shortest",
    OUTPUT_VIDEO
]

print("Rebuilding:", OUTPUT_VIDEO)
subprocess.run(cmd, check=True)

print("Created:", OUTPUT_VIDEO)
print("Size (MB):", round(os.path.getsize(OUTPUT_VIDEO) / 1024**2, 2))

## 9. Download the finished video

In [ ]:
from IPython.display import FileLink, display

display(FileLink(OUTPUT_VIDEO, result_html_prefix="Download upscaled video: "))

## Optional cleanup

Run this **only after downloading and checking the output**. It deletes the temporary PNG frames to free Kaggle storage.

In [ ]:
import shutil

if os.path.isdir(WORK_DIR):
    shutil.rmtree(WORK_DIR)
    print("Temporary frames deleted.")